# EgoVLP error-recognition baselines

Trains the V1 (MLP) and V2 (Transformer) error-recognition baselines on EgoVLP features,
on both the `step` and `recordings` splits, for four experiments in total.

Requires EgoVLP features extracted at a **1-second** window and stride
(`colab_feature_extraction.ipynb`). The dataloader indexes feature rows with raw
annotation timestamps in seconds, so any other stride silently misaligns the index and
returns empty slices for the later steps of each recording.

Run the cells in order from a fresh runtime.

In [1]:
import os
import sys

# --- COLAB SETUP ---
try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")
    drive.mount('/content/drive')

    REPO_URL = "https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git"
    BRANCH = "main"

    if not os.path.exists('/content/code'):
        !git clone --recursive {REPO_URL} /content/code
    os.chdir('/content/code')
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

    if '/content/code' not in sys.path:
        sys.path.append('/content/code')

    !pip install -q torcheval loguru einops ftfy timm transformers
    print(f"Working directory: {os.getcwd()}")
except ImportError:
    IN_COLAB = False
    print("Running locally")

Running in Google Colab
Mounted at /content/drive
Cloning into '/content/code'...
remote: Enumerating objects: 895, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 895 (delta 111), reused 107 (delta 95), pack-reused 750 (from 1)
Receiving objects: 100% (895/895), 96.52 MiB | 46.71 MiB/s, done.
Resolving deltas: 100% (485/485), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 14.96 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'
From https:/

## Part 1: Adapt Features Extraction Code (EgoVLP)

This section adapts the CaptainCook4D feature-extraction code to a new backbone, **EgoVLP**. Training below consumes features produced by that pipeline; the executable extraction runs in `colab_feature_extraction.ipynb`.

### 1.1 EgoVLP Model Definitions
We use the official EgoVLP implementation key components.

In [2]:
# --- FEATURE EXTRACTION ADAPTATION: EgoVLP ---
#
# EgoVLP (Ego-centric Video-Language Pretraining) - NeurIPS 2022
# Key characteristics:
# - Input: Video frames (224x224, 4 frames sampled per segment)
# - Architecture: SpaceTimeTransformer (TimeSformer-based)
# - Output: 256-dimensional aligned video-text embeddings
# - Pre-trained on: Ego4D dataset with video-narration pairs
#
# The adaptation involves:
# 1. Loading EgoVLP checkpoint (egovlp.pth)
# 2. Using the video encoder branch for feature extraction
# 3. Saving features in .npz format compatible with CaptainCook4D dataloader

import torch
import torch.nn as nn

def get_egovlp_model(ckpt_path=None):
    """
    Documents the EgoVLP video encoder configuration used for feature extraction.

    The executable version lives in `colab_feature_extraction.ipynb`, which
    initialises SpaceTimeTransformer with the EgoVLP config, loads the checkpoint
    and runs the encoder in eval mode. This function only reports the settings.
    """
    print("EgoVLP Feature Extractor Configuration:")
    print("  - Input resolution: 224x224")
    print("  - Frames per segment: 4")
    print("  - Feature dimension: 256")
    print("  - Architecture: SpaceTimeTransformer")
    return None

def extract_features_for_video(video_path, model, segment_length=1):
    """
    Interface of the per-video extraction step, implemented in
    `colab_feature_extraction.ipynb`:

    1. Load video and sample frames at segment_length intervals
    2. Resize and normalize with ImageNet statistics
    3. Pass through the EgoVLP video encoder
    4. Return features as a numpy array [T, 256]
    """
    raise NotImplementedError(
        "Feature extraction runs in colab_feature_extraction.ipynb")

# Show the adaptation is ready
print("EgoVLP adapter configuration reported. Training below uses the extracted features.")

EgoVLP adapter loaded. Pre-extracted features will be used for training.


### 1.2 Data Preparation (Feature Loading)

Since we have pre-extracted features in Google Drive, we will copy them to the local environment for high-speed training.

**Source**: `/content/drive/MyDrive/.../features/egovlp`  
**Destination**: `data/features/egovlp`

In [3]:
import os
import shutil
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
# Adjust this path to where your features are stored in Drive
DRIVE_FEATURES_PATH = "/content/drive/MyDrive/AML_Project/features/egovlp"
LOCAL_FEATURES_DIR = "data/features/egovlp"

os.makedirs(LOCAL_FEATURES_DIR, exist_ok=True)

# Copy features
if os.path.exists(DRIVE_FEATURES_PATH):
    print(f"Copying features from {DRIVE_FEATURES_PATH} to {LOCAL_FEATURES_DIR} ...")
    files = [f for f in os.listdir(DRIVE_FEATURES_PATH) if f.endswith('.npz')]
    for f in tqdm(files):
        src = os.path.join(DRIVE_FEATURES_PATH, f)
        dst = os.path.join(LOCAL_FEATURES_DIR, f)
        if not os.path.exists(dst):
            shutil.copy(src, dst)
    print("Feature copy complete.")
else:
    print(f"WARNING: Drive path {DRIVE_FEATURES_PATH} not found.")
    print("Please update DRIVE_FEATURES_PATH in this cell.")

Copying features from /content/drive/MyDrive/AML_Project/features/egovlp to data/features/egovlp ...


  0%|          | 0/384 [00:00<?, ?it/s]

Feature copy complete.


## Part 2: Train Baselines with EgoVLP

Now that we have the features, we verify the **EgoVLP backbone** by training the standard V1 (MLP) and V2 (Transformer) baselines.

**Configuration:**
- **Backbone**: `egovlp`
- **Feature Dimension**: 256 (EgoVLP standard)
- **Training**: 15 Epochs
- **Split**: `step` (threshold=0.6) and `recordings` (threshold=0.5)

In [4]:
# ============================================================================
# NOTEBOOK-COMPATIBLE CONFIG CLASS
# ============================================================================
# The original Config class uses ArgumentParser which fails in Jupyter/Colab.
# We create a notebook-friendly version that bypasses argument parsing.

import torch
import numpy as np
import random
from constants import Constants as const

class NotebookConfig:
    """Notebook-compatible configuration class (no ArgumentParser)."""

    def __init__(self):
        # Default values
        self.backbone = const.EGOVLP
        self.modality = [const.VIDEO]
        self.phase = "train"
        self.segment_length = 1
        self.segment_features_directory = "data"
        self.ckpt_directory = "checkpoints/"
        self.split = const.STEP_SPLIT
        self.batch_size = 32
        self.test_batch_size = 1
        self.num_epochs = 15
        self.lr = 1e-3
        self.weight_decay = 1e-3
        self.log_interval = 5
        self.dry_run = False
        self.ckpt = None
        self.seed = 42
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.variant = const.MLP_VARIANT
        self.model_name = None
        self.task_name = const.ERROR_RECOGNITION
        self.error_category = None
        self.enable_wandb = False
        self.save_model = True
        self.pos_weight = 2.5
        self.threshold = 0.6

    @property
    def args(self):
        """Live view of the settings; base.py prints this when building the loaders."""
        return {k: v for k, v in self.__dict__.items()}

    def print_config(self):
        print("=" * 60)
        print("CONFIGURATION")
        print("=" * 60)
        for k, v in self.__dict__.items():
            if k != 'args':
                print(f"  {k}: {v}")
        print("=" * 60)

# Set deterministic behavior
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"NotebookConfig class ready.")

Using device: cuda
NotebookConfig class ready.


### 2.1 Experiment 1: EgoVLP + MLP (V1)

In [6]:
# ============================================================================
# EXPERIMENT 1: EgoVLP + MLP (V1) on STEP Split
# ============================================================================
from base import train_model_base, train_step_test_step_dataset_base

# Initialize Config for MLP
conf_mlp = NotebookConfig()
conf_mlp.backbone = "egovlp"
conf_mlp.variant = "MLP"
conf_mlp.task_name = "error_recognition"
conf_mlp.segment_features_directory = "data"
conf_mlp.num_epochs = 15
conf_mlp.batch_size = 32
conf_mlp.lr = 1e-3
conf_mlp.weight_decay = 1e-3
conf_mlp.pos_weight = 2.5
conf_mlp.enable_wandb = False
conf_mlp.device = device
conf_mlp.split = "step"
conf_mlp.threshold = 0.6
conf_mlp.modality = ["video"]
conf_mlp.error_category = None

# Print configuration
conf_mlp.print_config()

# Load Data
print("\nLoading datasets...")
train_loader, val_loader, test_loader = train_step_test_step_dataset_base(conf_mlp)

# Train
print("\nStarting training...")
train_model_base(train_loader, val_loader, conf_mlp, test_loader=test_loader)
print("\nMLP Training Complete!")

CONFIGURATION
  backbone: egovlp
  modality: ['video']
  phase: train
  segment_length: 1
  segment_features_directory: data
  ckpt_directory: checkpoints/
  split: step
  batch_size: 32
  test_batch_size: 1
  num_epochs: 15
  lr: 0.001
  weight_decay: 0.001
  log_interval: 5
  dry_run: False
  ckpt: None
  seed: 42
  device: cuda
  variant: MLP
  model_name: None
  task_name: error_recognition
  error_category: None
  enable_wandb: False
  save_model: True
  pos_weight: 2.5
  threshold: 0.6

Loading datasets...
-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 32}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'backbone': 'egovlp', 'modality': ['video'], 'phase': 'train', 'segment_length': 1, 'segment_features_directory': 'data', 'ckpt_directory': 'checkpoints/', 'split': 'step', 'batch_size': 32, 'test_bat

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json

Starting training...


  0%|          | 0/118 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train Epoch: 1, Progress: 117/118, Loss: 1.011107: 100%|██████████| 118/118 [01:40<00:00,  1.18it/s]
  0%|          | 0/774 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader runnin

----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.356689852700491, 'recall': 0.2624764772299586, 'f1': 0.3024153332466068, 'accuracy': 0.5319192271880819, 'auc': np.float64(0.4762805254662152), 'pr_auc': tensor(0.3787)}
val Step Level Metrics: {'precision': 0.2892561983471074, 'recall': 0.14227642276422764, 'f1': 0.1907356948228883, 'accuracy': 0.6162790697674418, 'auc': np.float64(0.480729551613698), 'pr_auc': tensor(0.3138)}
----------------------------------------------------------------


...

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3502693965517241, 'recall': 0.5484224734266914, 'f1': 0.42750049319392386, 'accuracy': 0.5888159448354036, 'auc': np.float64(0.607222288123058), 'pr_auc': tensor(0.3185)}
test Step Level Metrics: {'precision': 0.44274809160305345, 'recall': 0.23293172690763053, 'f1': 0.30526315789473685, 'accuracy': 0.6691729323308271, 'auc': np.float64(0.656374130401387), 'pr_auc': tensor(0.3425)}
----------------------------------------------------------------
Epoch: 14, Train Loss: 0.874107, Val Loss: 1.069104, Test Loss: 1.029179, Val Precision: 0.365079, Val Recall: 0.186992, Val F1: 0.247312, Val AUC: 0.596876


  0%|          | 0/118 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train Epoch: 15, Progress: 117/118, Loss: 1.090789: 100%|██████████| 118/118 [01:37<00:00,  1.22it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:23<00:00, 32.77it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.43887930521682234, 'recall': 0.5553631915694393, 'f1': 0.4902977139819245, 'accuracy': 0.5536545623836127, 'auc': np.float64(0.5750011920375427), 'pr_auc': tensor(0.4156)}
val Step Level Metrics: {'precision': 0.4, 'recall': 0.3008130081300813, 'f1': 0.3433874709976798, 'accuracy': 0.6343669250645995, 'auc': np.float64(0.5958441118502094), 'pr_auc': tensor(0.3425)}
----------------------------------------------------------------


  0%|          | 0/798 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 42346/798: 100%|██████████| 798/798 [00:23<00:00, 33.61it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3515108352833452, 'recall': 0.5829255947359541, 'f1': 0.43856308707793856, 'accuracy': 0.5822037500590375, 'auc': np.float64(0.6195854287973174), 'pr_auc': tensor(0.3217)}
test Step Level Metrics: {'precision': 0.48056537102473496, 'recall': 0.5461847389558233, 'f1': 0.5112781954887218, 'accuracy': 0.6741854636591479, 'auc': np.float64(0.6977710477611723), 'pr_auc': tensor(0.4041)}
----------------------------------------------------------------
Epoch: 15, Train Loss: 0.877013, Val Loss: 1.061833, Test Loss: 1.001343, Val Precision: 0.400000, Val Recall: 0.300813, Val F1: 0.343387, Val AUC: 0.595844

MLP Training Complete!


### 2.2 Experiment 2: EgoVLP + Transformer (V2)

In [7]:
# ============================================================================
# EXPERIMENT 2: EgoVLP + Transformer (V2) on STEP Split
# ============================================================================
from base import train_model_base, train_step_test_step_dataset_base

# Initialize Config for Transformer
conf_tf = NotebookConfig()
conf_tf.backbone = "egovlp"
conf_tf.variant = "Transformer"
conf_tf.task_name = "error_recognition"
conf_tf.segment_features_directory = "data"
conf_tf.num_epochs = 15
conf_tf.batch_size = 32
conf_tf.lr = 1e-4  # Lower LR for Transformer stability
conf_tf.weight_decay = 1e-3
conf_tf.pos_weight = 2.5
conf_tf.enable_wandb = False
conf_tf.device = device
conf_tf.split = "step"
conf_tf.threshold = 0.6
conf_tf.modality = ["video"]
conf_tf.error_category = None

# Print configuration
conf_tf.print_config()

# Load Data
print("\nLoading datasets...")
train_loader, val_loader, test_loader = train_step_test_step_dataset_base(conf_tf)

# Train
print("\nStarting training...")
train_model_base(train_loader, val_loader, conf_tf, test_loader=test_loader)
print("\nTransformer Training Complete!")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


CONFIGURATION
  backbone: egovlp
  modality: ['video']
  phase: train
  segment_length: 1
  segment_features_directory: data
  ckpt_directory: checkpoints/
  split: step
  batch_size: 32
  test_batch_size: 1
  num_epochs: 15
  lr: 0.0001
  weight_decay: 0.001
  log_interval: 5
  dry_run: False
  ckpt: None
  seed: 42
  device: cuda
  variant: Transformer
  model_name: None
  task_name: error_recognition
  error_category: None
  enable_wandb: False
  save_model: True
  pos_weight: 2.5
  threshold: 0.6

Loading datasets...
-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 32}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'backbone': 'egovlp', 'modality': ['video'], 'phase': 'train', 'segment_length': 1, 'segment_features_directory': 'data', 'ckpt_directory': 'checkpoints/', 'split': 'step', 'batch_size': 32, 

  0%|          | 0/118 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train Epoch: 1, Progress: 117/118, Loss: 1.143020: 100%|██████████| 118/118 [01:41<00:00,  1.17it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:21<00:00, 35.48it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.38556481105241774, 'recall': 0.5713963116296575, 'f1': 0.4604373275103873, 'accuracy': 0.48233822160148976, 'auc': np.float64(0.5023848337764119), 'pr_auc': tensor(0.3860)}
val Step Level Metrics: {'precision': 0.35398230088495575, 'recall': 0.3252032520325203, 'f1': 0.3389830508474576, 'accuracy': 0.5968992248062015, 'auc': np.float64(0.5096236757822123), 'pr_auc': tensor(0.3296)}
----------------------------------------------------------------


  0%|          | 0/798 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 42346/798: 100%|██████████| 798/798 [00:24<00:00, 33.20it/s]


...

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.35384767361454117, 'recall': 0.604352961025814, 'f1': 0.4463551401869159, 'accuracy': 0.5803145515515042, 'auc': np.float64(0.6225310066476072), 'pr_auc': tensor(0.3246)}
test Step Level Metrics: {'precision': 0.47586206896551725, 'recall': 0.5542168674698795, 'f1': 0.5120593692022264, 'accuracy': 0.6704260651629073, 'auc': np.float64(0.6851376361548197), 'pr_auc': tensor(0.4028)}
----------------------------------------------------------------
Epoch: 14, Train Loss: 0.800934, Val Loss: 1.094216, Test Loss: 1.000127, Val Precision: 0.406699, Val Recall: 0.345528, Val F1: 0.373626, Val AUC: 0.581794


  0%|          | 0/118 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train Epoch: 15, Progress: 117/118, Loss: 0.873552: 100%|██████████| 118/118 [01:35<00:00,  1.23it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:22<00:00, 34.19it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.42938644261189773, 'recall': 0.6231840421528039, 'f1': 0.5084443898544494, 'accuracy': 0.534217877094972, 'auc': np.float64(0.5735305542116806), 'pr_auc': tensor(0.4132)}
val Step Level Metrics: {'precision': 0.375609756097561, 'recall': 0.3130081300813008, 'f1': 0.34146341463414637, 'accuracy': 0.6162790697674418, 'auc': np.float64(0.5950434220251294), 'pr_auc': tensor(0.3359)}
----------------------------------------------------------------


  0%|          | 0/798 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 42346/798: 100%|██████████| 798/798 [00:24<00:00, 32.52it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3485046098493366, 'recall': 0.653703391260334, 'f1': 0.4546334594737305, 'accuracy': 0.5609738818306333, 'auc': np.float64(0.6293495353996603), 'pr_auc': tensor(0.3248)}
test Step Level Metrics: {'precision': 0.4723127035830619, 'recall': 0.5823293172690763, 'f1': 0.5215827338129496, 'accuracy': 0.6666666666666666, 'auc': np.float64(0.7000387707478365), 'pr_auc': tensor(0.4054)}
----------------------------------------------------------------
Epoch: 15, Train Loss: 0.791149, Val Loss: 1.079578, Test Loss: 1.000449, Val Precision: 0.375610, Val Recall: 0.313008, Val F1: 0.341463, Val AUC: 0.595043

Transformer Training Complete!


### 2.3 Experiment 3: EgoVLP + MLP on RECORDINGS Split

In [8]:
# ============================================================================
# EXPERIMENT 3: EgoVLP + MLP (V1) on RECORDINGS Split
# ============================================================================
from base import train_model_base, train_step_test_step_dataset_base

conf_mlp_rec = NotebookConfig()
conf_mlp_rec.backbone = "egovlp"
conf_mlp_rec.variant = "MLP"
conf_mlp_rec.task_name = "error_recognition"
conf_mlp_rec.segment_features_directory = "data"
conf_mlp_rec.num_epochs = 15
conf_mlp_rec.batch_size = 32
conf_mlp_rec.lr = 1e-3
conf_mlp_rec.weight_decay = 1e-3
conf_mlp_rec.pos_weight = 2.5
conf_mlp_rec.enable_wandb = False
conf_mlp_rec.device = device
conf_mlp_rec.split = "recordings"
conf_mlp_rec.threshold = 0.5
conf_mlp_rec.modality = ["video"]
conf_mlp_rec.error_category = None

conf_mlp_rec.print_config()

print("\nLoading datasets...")
train_loader, val_loader, test_loader = train_step_test_step_dataset_base(conf_mlp_rec)

print("\nStarting training...")
train_model_base(train_loader, val_loader, conf_mlp_rec, test_loader=test_loader)
print("\nMLP (Recordings) Training Complete!")

CONFIGURATION
  backbone: egovlp
  modality: ['video']
  phase: train
  segment_length: 1
  segment_features_directory: data
  ckpt_directory: checkpoints/
  split: recordings
  batch_size: 32
  test_batch_size: 1
  num_epochs: 15
  lr: 0.001
  weight_decay: 0.001
  log_interval: 5
  dry_run: False
  ckpt: None
  seed: 42
  device: cuda
  variant: MLP
  model_name: None
  task_name: error_recognition
  error_category: None
  enable_wandb: False
  save_model: True
  pos_weight: 2.5
  threshold: 0.5

Loading datasets...
-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 32}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'backbone': 'egovlp', 'modality': ['video'], 'phase': 'train', 'segment_length': 1, 'segment_features_directory': 'data', 'ckpt_directory': 'checkpoints/', 'split': 'step', 'batch_size': 32, 'te

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json

Starting training...


  0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train Epoch: 1, Progress: 124/125, Loss: 1.139715: 100%|██████████| 125/125 [01:38<00:00,  1.26it/s]
val Progress: 37282/681: 100%|██████████| 681/681 [00:20<00:00, 32.80it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.34090487095376615, 'recall': 0.45423057892132607, 'f1': 0.38949192094190854, 'accuracy': 0.5368542460168446, 'auc': np.float64(0.522588362489089), 'pr_auc': tensor(0.3324)}
val Step Level Metrics: {'precision': 0.34502923976608185, 'recall': 0.502127659574468, 'f1': 0.4090121317157712, 'accuracy': 0.49926578560939794, 'auc': np.float64(0.503673313615113), 'pr_auc': tensor(0.3451)}
----------------------------------------------------------------


...

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.40044667027612346, 'recall': 0.46937966047913693, 'f1': 0.43218172522094805, 'accuracy': 0.5944387928111224, 'auc': np.float64(0.5882427904841738), 'pr_auc': tensor(0.3624)}
test Step Level Metrics: {'precision': 0.43828715365239296, 'recall': 0.7219917012448133, 'f1': 0.5454545454545454, 'accuracy': 0.5678092399403875, 'auc': np.float64(0.6311878799575412), 'pr_auc': tensor(0.4163)}
----------------------------------------------------------------
Epoch: 14, Train Loss: 0.833507, Val Loss: 1.112814, Test Loss: 1.105721, Val Precision: 0.407666, Val Recall: 0.497872, Val F1: 0.448276, Val AUC: 0.581481


  0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train Epoch: 15, Progress: 124/125, Loss: 0.904391: 100%|██████████| 125/125 [01:41<00:00,  1.23it/s]
val Progress: 37282/681: 100%|██████████| 681/681 [00:20<00:00, 33.47it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.3651510872634849, 'recall': 0.4265215239980208, 'f1': 0.39345758843666795, 'accuracy': 0.5722868944799099, 'auc': np.float64(0.5560199030653047), 'pr_auc': tensor(0.3423)}
val Step Level Metrics: {'precision': 0.4174757281553398, 'recall': 0.548936170212766, 'f1': 0.4742647058823529, 'accuracy': 0.580029368575624, 'auc': np.float64(0.5859937028909455), 'pr_auc': tensor(0.3848)}
----------------------------------------------------------------


  0%|          | 0/671 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 38337/671: 100%|██████████| 671/671 [00:23<00:00, 29.03it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.40086117091464685, 'recall': 0.4578772013326987, 'f1': 0.4274763932605073, 'accuracy': 0.5967081409604299, 'auc': np.float64(0.5890108712288693), 'pr_auc': tensor(0.3618)}
test Step Level Metrics: {'precision': 0.44416243654822335, 'recall': 0.7261410788381742, 'f1': 0.5511811023622047, 'accuracy': 0.5752608047690015, 'auc': np.float64(0.6309755862202066), 'pr_auc': tensor(0.4209)}
----------------------------------------------------------------
Epoch: 15, Train Loss: 0.824478, Val Loss: 1.114944, Test Loss: 1.110955, Val Precision: 0.417476, Val Recall: 0.548936, Val F1: 0.474265, Val AUC: 0.585994

MLP (Recordings) Training Complete!


### 2.4 Experiment 4: EgoVLP + Transformer on RECORDINGS Split

In [9]:
# ============================================================================
# EXPERIMENT 4: EgoVLP + Transformer (V2) on RECORDINGS Split
# ============================================================================
from base import train_model_base, train_step_test_step_dataset_base

conf_tf_rec = NotebookConfig()
conf_tf_rec.backbone = "egovlp"
conf_tf_rec.variant = "Transformer"
conf_tf_rec.task_name = "error_recognition"
conf_tf_rec.segment_features_directory = "data"
conf_tf_rec.num_epochs = 15
conf_tf_rec.batch_size = 32
conf_tf_rec.lr = 1e-4
conf_tf_rec.weight_decay = 1e-3
conf_tf_rec.pos_weight = 2.5
conf_tf_rec.enable_wandb = False
conf_tf_rec.device = device
conf_tf_rec.split = "recordings"
conf_tf_rec.threshold = 0.5
conf_tf_rec.modality = ["video"]
conf_tf_rec.error_category = None

conf_tf_rec.print_config()

print("\nLoading datasets...")
train_loader, val_loader, test_loader = train_step_test_step_dataset_base(conf_tf_rec)

print("\nStarting training...")
train_model_base(train_loader, val_loader, conf_tf_rec, test_loader=test_loader)
print("\nTransformer (Recordings) Training Complete!")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


CONFIGURATION
  backbone: egovlp
  modality: ['video']
  phase: train
  segment_length: 1
  segment_features_directory: data
  ckpt_directory: checkpoints/
  split: recordings
  batch_size: 32
  test_batch_size: 1
  num_epochs: 15
  lr: 0.0001
  weight_decay: 0.001
  log_interval: 5
  dry_run: False
  ckpt: None
  seed: 42
  device: cuda
  variant: Transformer
  model_name: None
  task_name: error_recognition
  error_category: None
  enable_wandb: False
  save_model: True
  pos_weight: 2.5
  threshold: 0.5

Loading datasets...
-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 32}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'backbone': 'egovlp', 'modality': ['video'], 'phase': 'train', 'segment_length': 1, 'segment_features_directory': 'data', 'ckpt_directory': 'checkpoints/', 'split': 'step', 'batch_size'

  0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train Epoch: 1, Progress: 124/125, Loss: 2.069156: 100%|██████████| 125/125 [01:41<00:00,  1.23it/s]
val Progress: 37282/681: 100%|██████████| 681/681 [00:21<00:00, 32.31it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.3390891089108911, 'recall': 0.3530430479960416, 'f1': 0.3459254171548624, 'accuracy': 0.5657690038088086, 'auc': np.float64(0.5211616491486657), 'pr_auc': tensor(0.3301)}
val Step Level Metrics: {'precision': 0.3743961352657005, 'recall': 0.6595744680851063, 'f1': 0.4776579352850539, 'accuracy': 0.5022026431718062, 'auc': np.float64(0.5283369907451579), 'pr_auc': tensor(0.3644)}
----------------------------------------------------------------


  0%|          | 0/671 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 38337/671: 100%|██████████| 671/671 [00:21<00:00, 31.04it/s]


...

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.39904002603319233, 'recall': 0.3891004283674441, 'f1': 0.39400755080729377, 'accuracy': 0.606437645094817, 'auc': np.float64(0.5900771715847283), 'pr_auc': tensor(0.3561)}
test Step Level Metrics: {'precision': 0.4318840579710145, 'recall': 0.6182572614107884, 'f1': 0.5085324232081911, 'accuracy': 0.5707898658718331, 'auc': np.float64(0.6142140306860947), 'pr_auc': tensor(0.4041)}
----------------------------------------------------------------
Epoch: 14, Train Loss: 0.784743, Val Loss: 1.192136, Test Loss: 1.175689, Val Precision: 0.386997, Val Recall: 0.531915, Val F1: 0.448029, Val AUC: 0.574945


  0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train Epoch: 15, Progress: 124/125, Loss: 1.054263: 100%|██████████| 125/125 [01:40<00:00,  1.24it/s]
val Progress: 37282/681: 100%|██████████| 681/681 [00:22<00:00, 30.55it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.36838434039584966, 'recall': 0.3484248721754907, 'f1': 0.3581267217630854, 'accuracy': 0.5937717933587254, 'auc': np.float64(0.5531961739677942), 'pr_auc': tensor(0.3403)}
val Step Level Metrics: {'precision': 0.4074074074074074, 'recall': 0.5617021276595745, 'f1': 0.47227191413237923, 'accuracy': 0.566813509544787, 'auc': np.float64(0.5877778837897147), 'pr_auc': tensor(0.3801)}
----------------------------------------------------------------


  0%|          | 0/671 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 38337/671: 100%|██████████| 671/671 [00:21<00:00, 31.22it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.40198361362656315, 'recall': 0.3697445660796446, 'f1': 0.3851906945993967, 'accuracy': 0.6118892975454522, 'auc': np.float64(0.5862944420887648), 'pr_auc': tensor(0.3559)}
test Step Level Metrics: {'precision': 0.4686192468619247, 'recall': 0.46473029045643155, 'f1': 0.4666666666666667, 'accuracy': 0.6184798807749627, 'auc': np.float64(0.6147158158834314), 'pr_auc': tensor(0.4100)}
----------------------------------------------------------------
Epoch: 15, Train Loss: 0.790621, Val Loss: 1.188077, Test Loss: 1.194240, Val Precision: 0.407407, Val Recall: 0.561702, Val F1: 0.472272, Val AUC: 0.587778

Transformer (Recordings) Training Complete!


---

## 3. Results summary (final epoch)

### Step-Level Metrics (final epoch, test)

| Model | Split | LR | Threshold | F1 | AUC | Accuracy | Precision | Recall |
|-------|-------|-----|-----------|-----|-----|----------|-----------|--------|
| MLP | step | 1e-3 | 0.6 | 51.13% | 69.78% | 67.42% | 48.06% | 54.62% |
| Transformer | step | 1e-4 | 0.6 | 52.06% | 70.00% | 66.54% | 47.08% | 58.23% |
| MLP | recordings | 1e-3 | 0.5 | 55.12% | 63.10% | 57.53% | 44.42% | 72.61% |
| Transformer | recordings | 1e-4 | 0.5 | 46.25% | 61.45% | 61.55% | 46.44% | 46.06% |

### Sub-Step-Level Metrics (final epoch, test)

| Model | Split | F1 | AUC | Accuracy | Precision | Recall |
|-------|-------|-----|-----|----------|-----------|--------|
| MLP | step | 43.86% | 61.96% | 58.22% | 35.15% | 58.29% |
| Transformer | step | 45.47% | 62.94% | 56.11% | 34.86% | 65.35% |
| MLP | recordings | 42.75% | 58.90% | 59.67% | 40.09% | 45.79% |
| Transformer | recordings | 38.60% | 58.63% | 61.23% | 40.27% | 37.07% |

These are final-epoch figures. Section 4 re-evaluates each checkpoint selected on
validation AUC, and those are the numbers the report quotes.

### Comparison with published baselines

Published CaptainCook4D Omnivore results (repo `README.md`, Table 2) alongside this run.

| Split | Model | Published F1 (Omnivore) | This run F1 (EgoVLP) | Published AUC | This run AUC |
|-------|-------|------------------------|---------------------|---------------|--------------|
| step | MLP | 24.26% | **51.13%** | 75.74% | 69.78% |
| step | Transformer | 55.39% | 52.06% | 75.62% | 70.00% |
| recordings | MLP | 55.42% | **55.12%** | 63.03% | **63.10%** |
| recordings | Transformer | 40.73% | **46.25%** | 62.27% | 61.45% |

EgoVLP is competitive with the released Omnivore features: it beats the published MLP on
step-split F1 (+26.9) and the Transformer on recordings-split F1 (+5.5), ties on the
recordings/MLP cell (F1 within 0.3 points, AUC within 0.07), and trails by about 6 points
on step-split AUC.

### Remaining caveats

1. **Preprocessing is asymmetric.** Per-recording standardization and +/-10 clipping are
   applied in the EgoVLP dataloader branch only, not to Omnivore/SlowFast. Any backbone
   comparison currently mixes a backbone change with a preprocessing change.
2. **The Omnivore column is the published number, not a local re-run.** A strictly fair
   comparison requires running Omnivore through this same pipeline.
3. **MLP and Transformer use different learning rates** (`1e-3` vs `1e-4`). The lower
   Transformer LR is a deliberate stability choice, but V1-vs-V2 claims should state it.
   The MLP is sensitive to this: at `1e-5` it reaches only 24.29% step-split F1.
4. Metrics are final-epoch, single-seed, 15 epochs. No error bars.

---
## 4. Test-set evaluation

`train_model_base` selects the best checkpoint by **validation** AUC and prints validation
metrics each epoch, so the per-epoch `Precision / Recall / F1 / AUC` figures above are
validation numbers, not test numbers. (The `Test Loss` field on the same line is genuinely
the test loss - only the loss comes from the test split.)

Reporting the best validation AUC as the result would report a score on the same split that
chose the checkpoint, which is optimistic. This section loads each `*_best.pt` checkpoint
and evaluates it **once** on the held-out test split. These are the numbers to report.

No retraining is required: `train_model_base` already wrote the checkpoints.

In [10]:
# Evaluate each selected checkpoint once on the held-out test split.
import os
import torch
import torch.nn as nn
from base import fetch_model, fetch_model_name, test_er_model, train_step_test_step_dataset_base


def evaluate_on_test(conf):
    """Load {model_name}_best.pt (selected on validation AUC) and score it on test."""
    model_name = fetch_model_name(conf)
    ckpt_path = os.path.join(conf.ckpt_directory, conf.task_name, conf.variant,
                             conf.backbone, f"{model_name}_best.pt")
    if not os.path.exists(ckpt_path):
        print(f"  MISSING: {ckpt_path}")
        return None

    _, _, test_loader = train_step_test_step_dataset_base(conf)

    model = fetch_model(conf).to(conf.device)
    model.load_state_dict(torch.load(ckpt_path, map_location=conf.device))
    model.eval()

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([conf.pos_weight], device=conf.device))
    _, _, step_metrics = test_er_model(
        model, test_loader, criterion, conf.device, phase='test',
        threshold=getattr(conf, 'threshold', 0.6))
    return step_metrics


experiments = [("MLP / step", conf_mlp),
               ("Transformer / step", conf_tf),
               ("MLP / recordings", conf_mlp_rec),
               ("Transformer / recordings", conf_tf_rec)]

print(f"{'experiment':<26}{'thr':>6}{'P':>9}{'R':>9}{'F1':>9}{'AUC':>9}")
print("-" * 68)
for name, conf in experiments:
    m = evaluate_on_test(conf)
    if m is None:
        continue
    print(f"{name:<26}{getattr(conf, 'threshold', 0.6):>6.2f}"
          f"{m['precision']*100:>8.2f}%{m['recall']*100:>8.2f}%"
          f"{m['f1']*100:>8.2f}%{m['auc']*100:>8.2f}%")

print("\nThese are test-split numbers from checkpoints selected on validation AUC,")
print("and are the ones to report. The per-epoch figures above are validation metrics.")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


experiment                   thr        P        R       F1      AUC
--------------------------------------------------------------------
-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 32}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'backbone': 'egovlp', 'modality': ['video'], 'phase': 'train', 'segment_length': 1, 'segment_features_directory': 'data', 'ckpt_directory': 'checkpoints/', 'split': 'step', 'batch_size': 32, 'test_batch_size': 1, 'num_epochs': 15, 'lr': 0.001, 'weight_decay': 0.001, 'log_interval': 5, 'dry_run': False, 'ckpt': None, 'seed': 42, 'device': 'cuda', 'variant': 'MLP', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'enable_wandb': False, 'save_model': True, 'pos_weight': 2.5, 'threshold': 0.6}
--------------------------------------------------------

  0%|          | 0/798 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 42346/798: 100%|██████████| 798/798 [00:23<00:00, 33.30it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potentia

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3515108352833452, 'recall': 0.5829255947359541, 'f1': 0.43856308707793856, 'accuracy': 0.5822037500590375, 'auc': np.float64(0.6195854287973174), 'pr_auc': tensor(0.3217)}
test Step Level Metrics: {'precision': 0.48056537102473496, 'recall': 0.5461847389558233, 'f1': 0.5112781954887218, 'accuracy': 0.6741854636591479, 'auc': np.float64(0.6977710477611723), 'pr_auc': tensor(0.4041)}
----------------------------------------------------------------
MLP / step                  0.60   48.06%   54.62%   51.13%   69.78%
-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 32}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'backbone': 'egovlp', 'modality': ['video'], 'phase': 'train', 'segment_length': 1, 'segm

  0%|          | 0/798 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 42346/798: 100%|██████████| 798/798 [00:23<00:00, 33.45it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potentia

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3484616078679058, 'recall': 0.6515943985152691, 'f1': 0.4540858318636096, 'accuracy': 0.5614225664761725, 'auc': np.float64(0.6305523543021051), 'pr_auc': tensor(0.3246)}
test Step Level Metrics: {'precision': 0.46579804560260585, 'recall': 0.5742971887550201, 'f1': 0.5143884892086331, 'accuracy': 0.6616541353383458, 'auc': np.float64(0.6999436726871053), 'pr_auc': tensor(0.4003)}
----------------------------------------------------------------
Transformer / step          0.60   46.58%   57.43%   51.44%   69.99%
-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 32}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'backbone': 'egovlp', 'modality': ['video'], 'phase': 'train', 'segment_length': 1, 'segme

  0%|          | 0/671 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 38337/671: 100%|██████████| 671/671 [00:20<00:00, 33.05it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potentia

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.40086117091464685, 'recall': 0.4578772013326987, 'f1': 0.4274763932605073, 'accuracy': 0.5967081409604299, 'auc': np.float64(0.5890108712288693), 'pr_auc': tensor(0.3618)}
test Step Level Metrics: {'precision': 0.44416243654822335, 'recall': 0.7261410788381742, 'f1': 0.5511811023622047, 'accuracy': 0.5752608047690015, 'auc': np.float64(0.6309755862202066), 'pr_auc': tensor(0.4209)}
----------------------------------------------------------------
MLP / recordings            0.50   44.42%   72.61%   55.12%   63.10%
-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 32}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'backbone': 'egovlp', 'modality': ['video'], 'phase': 'train', 'segment_length': 1, 'segm

  0%|          | 0/671 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test Progress: 38337/671: 100%|██████████| 671/671 [00:22<00:00, 30.40it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.405176144724314, 'recall': 0.3713311121688085, 'f1': 0.3875160395711743, 'accuracy': 0.6140282233873282, 'auc': np.float64(0.5878549079893599), 'pr_auc': tensor(0.3572)}
test Step Level Metrics: {'precision': 0.47435897435897434, 'recall': 0.4605809128630705, 'f1': 0.4673684210526316, 'accuracy': 0.6229508196721312, 'auc': np.float64(0.6154491942487695), 'pr_auc': tensor(0.4122)}
----------------------------------------------------------------
Transformer / recordings    0.50   47.44%   46.06%   46.74%   61.54%

These are test-split numbers from checkpoints selected on validation AUC,
and are the ones to report. The per-epoch figures above are validation metrics.
